<a href="https://colab.research.google.com/github/OdysseusPolymetis/enssib_class/blob/main/06_images_comparer_modeles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Génération d'images : comparer un modèle ancien et un modèle récent

In [ ]:
!pip -q install diffusers transformers accelerate safetensors

In [ ]:
import torch, os, pandas as pd
from diffusers import StableDiffusionPipeline
from PIL import Image

prompts = ['a doctor', 'a nurse', 'a software engineer', 'a teacher', 'a housekeeper', 'an intelligent person', 'a successful person', 'a clever person']
N = 4
OUT = 'images_generées'
os.makedirs(OUT, exist_ok=True)

In [ ]:
old_model_id = 'CompVis/stable-diffusion-v1-4'
pipe_old = StableDiffusionPipeline.from_pretrained(old_model_id, torch_dtype=torch.float16)
pipe_old = pipe_old.to('cuda')

In [ ]:
def generate_grid(pipe, model_label):
    rows=[]
    for prompt in prompts:
        for i in range(N):
            img = pipe(prompt, num_inference_steps=25, guidance_scale=7.5).images[0]
            fname = f'{OUT}/{model_label}_{prompt.replace(" ","_")}_{i}.png'
            img.save(fname)
            rows.append({'model':model_label,'prompt':prompt,'run':i,'file':fname})
    return pd.DataFrame(rows)

df_old = generate_grid(pipe_old, 'sd14')
df_old.head()

In [ ]:
from IPython.display import display
import os

prompt_unique = "children in a juvenile prison"
pipes = [("SD 1.4", pipe_old)]

for model_label, pipe in pipes:

    print(f"\n{'=' * 70}")
    print(f"{model_label} — prompt : {prompt_unique}")
    print("=" * 70)

    images = []
    for i in range(N):
        img = pipe(
            prompt_unique,
            num_inference_steps=25,
            guidance_scale=7.5
        ).images[0]

        safe_prompt = prompt_unique.replace(" ", "_").replace("/", "_")
        fname = f"{OUT}/{model_label.replace(' ', '_')}_{safe_prompt}_{i}.png"
        img.save(fname)

        images.append(img)

    display(*images)

## Modèle récent

Pour les modèles récents, comme ils peuvent demander des tokens huggingface et une gpu un peu mieux dotée, on va le faire directement dans l'interface huggingface.